In [1]:
# =================== RQ2: DeepFM (CF) and DeepFM+Tags Hybrid ===================
# Requirements:
#    pip install libreco==1.* scikit-learn pandas numpy matplotlib tqdm
#    TensorFlow 1.x compatible backend (LibRecommender uses tf.compat.v1)
# ===============================================================================
import os, re, math, random
from itertools import product
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# Here, I am importing the necessary libraries for building and evaluating recommender systems.
# libreco.data is used for handling and preparing the data for the models.
# libreco.algorithms provides the DeepFM and ItemCF algorithms.
from libreco.data import DatasetPure, DatasetFeat, random_split
from libreco.algorithms import DeepFM, ItemCF  # ItemKNN baseline            
# Here, I am importing the necessary libraries to create and manipulate a DataFrame,
# as well as to interact with the operating system and to handle regular expressions,
# math operations, and random number generation.
from libreco.data import DatasetPure, DatasetFeat, random_split
from libreco.algorithms import DeepFM, ItemCF  # ItemKNN baseline 

Instructions for updating:
non-resource variables are not supported in the long term


In [2]:
# -------------------- SETTINGS --------------------
# Here, I am defining the directory where all the data files are stored.
DATA_DIR = r"C:\Users\GSI\Pictures\DATASETS"
# Here, I am creating the full file paths for the training and testing data, as well as the games metadata.
TRAIN_INTERACTIONS = os.path.join(DATA_DIR, "train_interactions.csv")
TEST_IN_FILE = os.path.join(DATA_DIR, "test_interactions_in.csv")
GAMES_FILE = os.path.join(DATA_DIR, "games.csv")

# Outputs (RQ2)
# Here, I am defining the file paths for saving the evaluation results and the final top-20 recommendations.
EVAL_SUMMARY_OUT = os.path.join(DATA_DIR, "rq2_eval_summary.csv")
EVAL_ALPHA_GRID_OUT = os.path.join(DATA_DIR, "rq2_alpha_sweep.csv")
OUTPUT_TOP20 = os.path.join(DATA_DIR, "rq2_top20_deepfm_hybrid.csv")
# Here, I am setting the directory where the plots will be saved.
PLOT_DIR = DATA_DIR

# Here, I am setting a random seed for reproducibility.
SEED = 42
np.random.seed(SEED); random.seed(SEED)

# Here, I am defining K, which is the number of top items to recommend (top-K).
K = 20
# Here, I am defining a grid of alpha values to sweep through for the hybrid model.
# Alpha controls the weight given to the collaborative filtering part of the recommendation.
ALPHAS_GRID = np.arange(0.0, 1.0, 0.05)

# gating / candidate settings (kept simple & memory-safe)
# Here, I am setting the number of candidate items to retrieve from the DeepFM model.
CAND_CF = 500                 # candidate width taken from model.recommend_user
# Here, I am setting the constant K for Reciprocal Rank Fusion (RRF).
RRF_K = 60                    # RRF constant for reciprocal-rank fusion

# content profile thresholds (adaptive alpha)
# Here, I am setting the minimum number of items with tags required to build a content profile.
MIN_TAG_ITEMS = 3
# Here, I am setting the minimum L2 norm for a user's content profile to be considered "strong."
MIN_PROFILE_NORM = 0.10
# Here, I am setting the sigmoid steepness parameter for the adaptive alpha calculation.
ADAPT_S = 2

# Here, I am getting the number of CPU cores available to configure TensorFlow.
CPU_CORES = max(1, os.cpu_count() or 1)
# Here, I am configuring the TensorFlow session to use all available CPU cores for parallel operations.
TF_SESS_CONFIG = {
    "intra_op_parallelism_threads": CPU_CORES,
    "inter_op_parallelism_threads": CPU_CORES,
}        

In [3]:
# -------------------- HELPERS --------------------
def load_csv(path):
    # Here, I am loading a CSV file into a pandas DataFrame.
    print(f"Loading {path} ...")
    # Here, I am reading the CSV and specifying `low_memory=False` to handle large files.
    df = pd.read_csv(path, low_memory=False)
    # Here, I am printing the number of rows and the column names to give a quick overview of the loaded data.
    print(f"Loaded {len(df):,} rows; columns={list(df.columns)[:8]}{'...' if df.shape[1]>8 else ''}")
    # Here, I am returning the loaded DataFrame.
    return df

def detect_columns(df):
    # Here, I am detecting the column names for user ID, item ID, and playtime/label.
    # Here, I am creating a list of lowercase column names for case-insensitive matching.
    cols = [c.lower() for c in df.columns]
    def pick(cands):
        # Here, I am iterating through a list of candidate column names to find a match in the DataFrame.
        for cand in cands:
            # Here, I am checking if a candidate name exists in the lowercase column list.
            if cand in cols: return df.columns[cols.index(cand)]
        # Here, I am returning None if no match is found.
        return None
    # Here, I am returning the detected column names as a tuple.
    return (
        pick(("user_id","user","userid","user id","steamid")),
        pick(("item_id","itemid","item","game_id","appid","app_id","id")),
        pick(("playtime","play_time","play_minutes","playtime_forever","rating","label"))
    )

# Here, I am compiling a regular expression to split strings by common delimiters.
DELIMS_RE = re.compile(r"[\,\|/;]+")
# Here, I am defining a dictionary of phrase synonyms to normalize tags and genres.
PHRASE_SYNONYMS = {
    "fps":"first_person_shooter","tps":"third_person_shooter",
    "rpg":"role_playing","arpg":"action_rpg",
    "roguelike":"rogue_like","rogue_like":"rogue_like","rogue_lite":"rogue_lite",
}
def normalize_phrase(s: str) -> str:
    # Here, I am normalizing a single string by converting it to lowercase and replacing spaces with underscores.
    s = s.strip().lower()
    # Here, I am replacing multiple spaces with a single one, and hyphens with underscores.
    s = re.sub(r"\s+"," ", s).replace("-", "_").replace(" ","_")
    # Here, I am removing any characters that are not a-z, 0-9, or an underscore.
    s = re.sub(r"[^a-z0-9_]+","", s)
    # Here, I am using the synonym dictionary to get a standardized phrase, or the original if not found.
    return PHRASE_SYNONYMS.get(s, s) if s else ""

def parse_tags_or_genres(raw: str) -> list:
    # Here, I am parsing a raw string of tags or genres into a list of unique, normalized tokens.
    # Here, I am handling cases where the input is missing or NaN.
    if pd.isna(raw): return []
    # Here, I am splitting the string by the defined delimiters and filtering out empty parts.
    parts = [p for p in DELIMS_RE.split(str(raw)) if p and p.strip()]
    # Here, I am initializing lists and sets to store tokens and track seen tokens.
    toks, seen = [], set()
    for p in parts:
        # Here, I am normalizing each part of the string.
        tok = normalize_phrase(p)
        # Here, I am ensuring the token is not empty and has not been seen before.
        if tok and tok not in seen:
            # Here, I am adding the new token to the set and list.
            seen.add(tok); toks.append(tok)
    # Here, I am returning the list of unique, normalized tokens.
    return toks

def sigmoid(x): 
    # Here, I am calculating the sigmoid function, which squashes a value to a range between 0 and 1.
    return 1.0/(1.0+np.exp(-x))

def recall_ndcg_hr_at_k(topk_items, gt_set, k=K):
    # Here, I am calculating recall, nDCG, and hit rate for a given list of recommended items.
    # Here, I am handling the case where the ground truth set is empty.
    if not gt_set: return 0.0, 0.0, 0.0
    # Here, I am creating a list of binary values (1 for a hit, 0 for a miss).
    hits = [1 if int(it) in gt_set else 0 for it in topk_items]
    # Here, I am calculating recall by dividing the number of hits by the total number of relevant items.
    recall = sum(hits) / float(len(gt_set))
    # Here, I am calculating hit rate, which is 1 if there's at least one hit, otherwise 0.
    hr = 1.0 if sum(hits) > 0 else 0.0
    # Here, I am calculating the Discounted Cumulative Gain (DCG).
    dcg = sum(rel / math.log2(r+2) for r, rel in enumerate(hits))
    # Here, I am calculating the Ideal Discounted Cumulative Gain (IDCG).
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, min(len(gt_set), k) + 1))
    # Here, I am calculating nDCG (normalized DCG).
    ndcg = dcg / idcg if idcg > 0 else 0.0
    # Here, I am returning the calculated metrics.
    return recall, ndcg, hr

def pairwise_cosine_from_tfidf(rows):
    # Here, I am calculating the average pairwise cosine distance (1 - similarity) for a set of items.
    # Here, I am checking if there are at least two items to compare.
    if rows.shape[0] < 2: return 0.0
    # Here, I am computing the pairwise cosine similarity matrix.
    sim = rows @ rows.T
    # Here, I am converting the sparse matrix to a dense NumPy array.
    sim = sim.toarray()
    # Here, I am getting the indices for the upper triangle of the similarity matrix, excluding the diagonal.
    iu = np.triu_indices(sim.shape[0], k=1)
    # Here, I am calculating the mean similarity and returning 1 minus that value, which represents diversity.
    return float(1.0 - sim[iu].mean())  # diversity = 1 - mean similarity

def score_to_rrf(scores, k_rrf=RRF_K):
    # Here, I am converting a list of scores into a list of RRF scores.
    # Here, I am converting the scores to a NumPy array.
    s = np.asarray(scores).astype(float).ravel()
    n = s.size
    # Here, I am handling the case of an empty score list.
    if n == 0: return s
    # Here, I am handling the case where all scores are the same.
    if np.all(np.isfinite(s)) and (s.max() == s.min()):
        return np.ones(n) / (k_rrf + 1.0)
    # Here, I am getting the ranks by sorting the scores in descending order.
    order = np.argsort(-s, kind="mergesort")
    # Here, I am creating an array for the RRF scores and filling it based on the ranks.
    rrf = np.zeros(n, dtype=float)
    for r, pos in enumerate(order):
        # Here, I am calculating the RRF score using the formula 1 / (k + rank).
        rrf[pos] = 1.0 / (k_rrf + r + 1.0)
    # Here, I am normalizing the RRF scores so the maximum value is 1.
    if rrf.max() > 0: rrf = rrf / rrf.max()
    # Here, I am returning the RRF scores.
    return rrf

def topk_from_scores(scores, allow_idx, k: int):
    # Here, I am selecting the top-K item indices based on scores, while excluding certain items.
    # Here, I am converting the scores and allowed indices to NumPy arrays.
    scores = np.asarray(scores).reshape(-1)
    allow_idx = np.asarray(allow_idx, dtype=int)
    # Here, I am handling the case where there are no allowed items.
    if allow_idx.size == 0: return np.array([], dtype=int)
    # Here, I am determining the number of items to take, which is the minimum of k and the number of allowed items.
    take = min(k, allow_idx.size)
    # Here, I am using `argpartition` to efficiently find the indices of the top-K items.
    part = np.argpartition(-scores[allow_idx], take - 1)[:take]
    # Here, I am getting the original indices of the candidate items.
    cand = allow_idx[part]
    # Here, I am sorting the top-K candidates by their scores to get the final ranked list.
    order = cand[np.argsort(-scores[cand])]
    # Here, I am returning the top-K indices.
    return order[:k]        

In [4]:
# -------------------- STEP 1: Load & clean (implicit) --------------------
# Here, I am starting the first step of the process and printing a header.
print("\n=== RQ2 / STEP 1: Load datasets ===")
# Here, I am loading the training interactions data from the specified CSV file.
inter_df = load_csv(TRAIN_INTERACTIONS)
# Here, I am detecting the user, item, and playtime columns in the DataFrame.
u_col, i_col, p_col = detect_columns(inter_df)
# Here, I am printing the detected column names to confirm they were found correctly.
print(f"Detected -> user: {u_col}, item: {i_col}, playtime/label: {p_col}")

# Here, I am renaming the columns to a standard 'user' and 'item' for consistency.
inter_df = inter_df.rename(columns={u_col:"user", i_col:"item"})
# Here, I am removing any rows that have missing user or item IDs.
inter_df = inter_df.dropna(subset=["user","item"])
# Here, I am ensuring the user and item IDs are integer types.
inter_df["user"] = inter_df["user"].astype(int)
inter_df["item"] = inter_df["item"].astype(int)
# implicit label: playtime > 0
# Here, I am handling the playtime/label column. If it exists, I'm converting it to a float.
if p_col: inter_df["playtime"] = inter_df[p_col].astype(float)
# Here, if the playtime column does not exist, I am assuming an implicit interaction and assigning a value of 1.0.
else:     inter_df["playtime"] = 1.0
# Here, I am creating a binary 'label' column: 1 if playtime is greater than 0, 0 otherwise.
inter_df["label"] = (inter_df["playtime"] > 0).astype(int)
# Here, I am removing duplicate user-item interactions, keeping only the last one.
inter_df = inter_df.drop_duplicates(["user","item"], keep="last")
# Here, I am printing the size of the cleaned dataset and the number of unique users and items.
print(f"After cleaning: {len(inter_df):,} rows, users={inter_df['user'].nunique():,}, items={inter_df['item'].nunique():,}")

# Here, I am loading the games metadata.
games_df = load_csv(GAMES_FILE)

# optional test_in for final top-20 export
# Here, I am checking if the test file for final recommendations exists.
test_in_df = None
if os.path.exists(TEST_IN_FILE):
    # Here, I am loading the test interactions data if the file is found.
    test_in_df = load_csv(TEST_IN_FILE)
    # Here, I am detecting the user and item columns and renaming them for consistency.
    tu, ti, _ = detect_columns(test_in_df)
    test_in_df = test_in_df.rename(columns={tu:'user', ti:'item'})
    # Here, I am ensuring the user and item IDs are integer types.
    test_in_df['user'] = test_in_df['user'].astype(int)
    test_in_df['item'] = test_in_df['item'].astype(int)      


=== RQ2 / STEP 1: Load datasets ===
Loading C:\Users\GSI\Pictures\DATASETS\train_interactions.csv ...
Loaded 2,293,985 rows; columns=['user_id', 'item_id', 'item_name', 'playtime']
Detected -> user: user_id, item: item_id, playtime/label: playtime
After cleaning: 2,267,389 rows, users=54,315, items=8,368
Loading C:\Users\GSI\Pictures\DATASETS\games.csv ...
Loaded 8,523 rows; columns=['item_id', 'item_name', 'publisher', 'genres', 'url', 'tags', 'sentiment', 'metascore']...
Loading C:\Users\GSI\Pictures\DATASETS\test_interactions_in.csv ...
Loaded 448,211 rows; columns=['user_id', 'item_id', 'item_name', 'playtime']


In [5]:
# -------------------- STEP 2: random split (0.8/0.1/0.1) --------------------
# Here, I am starting the second step and printing a header.
print("\n=== RQ2 / STEP 2: random_split ===")
# Here, I am randomly splitting the cleaned interactions data into training, evaluation, and test sets.
train_df, eval_df, test_df = random_split(inter_df[['user','item','label']], multi_ratios=[0.8,0.1,0.1], seed=SEED)
# Here, I am printing the size of each split to confirm the ratios.
print(f"Split -> train={len(train_df):,}, eval={len(eval_df):,}, test={len(test_df):,}")

# For coverage denominator (use original ids present in train+eval)
# Here, I am concatenating the training and evaluation sets to get the universe of all items seen during model development.
train_plus_eval_df = pd.concat([train_df, eval_df], ignore_index=True)
# Here, I am getting a sorted list of unique items from the combined training and evaluation sets.
all_items_universe = sorted(train_plus_eval_df['item'].unique().tolist())
# Here, I am creating a dictionary to map each item ID to its position in the sorted universe list.
item_universe_pos = {it:i for i,it in enumerate(all_items_universe)}    


=== RQ2 / STEP 2: random_split ===
Split -> train=1,813,911, eval=226,292, test=226,243


In [6]:
# -------------------- STEP 3: Build datasets --------------------
# Here, I am starting the third step and printing a header.
print("\n=== RQ2 / STEP 3: Build DatasetFeat (DeepFM) and DatasetPure (ItemCF) ===")
# DeepFM prefers DatasetFeat (even with only ID features)
# Here, I am building the training dataset for DeepFM, which uses `DatasetFeat`.
dfm_train_data, dfm_datainfo = DatasetFeat.build_trainset(train_df)
# Here, I am building the evaluation dataset for DeepFM.
dfm_eval_data = DatasetFeat.build_evalset(eval_df)
# Here, I am building the test dataset for DeepFM.
dfm_test_data = DatasetFeat.build_testset(test_df)

# ItemCF with DatasetPure
# Here, I am building the training dataset for ItemCF, which uses `DatasetPure` as it's a simple CF model.
icf_train_data, icf_datainfo = DatasetPure.build_trainset(train_df)
# Here, I am building the evaluation dataset for ItemCF.
icf_eval_data = DatasetPure.build_evalset(eval_df)
# Here, I am building the test dataset for ItemCF.
icf_test_data = DatasetPure.build_testset(test_df) 



=== RQ2 / STEP 3: Build DatasetFeat (DeepFM) and DatasetPure (ItemCF) ===


In [7]:
# -------------------- STEP 4: TF-IDF content (tags + genres) --------------------
# Here, I am starting the fourth step and printing a header.
print("\n=== RQ2 / STEP 4: TF-IDF over tags/genres (for hybrid & diversity) ===")
from sklearn.feature_extraction.text import TfidfVectorizer

# Here, I am creating a dictionary of lowercase column names to find the correct columns for item ID, tags, and genres.
cg = {c.lower(): c for c in games_df.columns}
# Here, I am detecting the column names for item ID, tags, and genres from the games metadata.
g_item = cg.get('item_id') or cg.get('appid') or cg.get('app_id') or cg.get('id')
g_tags = cg.get('tags') or cg.get('tag')
g_genre = cg.get('genres') or cg.get('genre')

# Here, I am creating a new DataFrame with only the item ID, tags, and genres columns.
meta = games_df[[g_item] + ([g_tags] if g_tags else []) + ([g_genre] if g_genre else [])].copy()
# Here, I am renaming the columns for clarity and consistency.
meta = meta.rename(columns={g_item:'item', g_tags:'tags', g_genre:'genres'})
# Here, I am ensuring the item ID is an integer type.
meta['item'] = meta['item'].astype(int)
# Here, I am filtering the metadata to include only the items present in the training and evaluation sets.
meta = meta[meta['item'].isin(all_items_universe)].copy()
# Here, I am parsing the raw tags and genres strings into lists of normalized tokens.
meta['toks_tags']   = meta['tags'].apply(parse_tags_or_genres)  if 'tags'   in meta.columns else [[]]
meta['toks_genres'] = meta['genres'].apply(parse_tags_or_genres) if 'genres' in meta.columns else [[]]
# Here, I am setting the item ID as the index for faster lookups.
meta_map = meta.set_index('item')

# Here, I am preparing the documents for TF-IDF vectorization.
content_docs = []
miss_tags = miss_gen = 0
for it in all_items_universe:
    # Here, I am looking up the item's metadata in the `meta_map` dictionary.
    if it in meta_map.index:
        # Here, I am getting the row for the current item.
        r = meta_map.loc[it]
        # Here, I am getting the parsed tags and genres.
        tt = r['toks_tags'] if 'toks_tags' in meta_map.columns else []
        gg = r['toks_genres'] if 'toks_genres' in meta_map.columns else []
    else:
        # Here, if the item is not in the metadata, I am using empty lists.
        tt, gg = [], []
    # Here, I am counting the items that are missing tags or genres.
    if not tt: miss_tags += 1
    if not gg: miss_gen  += 1
    # Here, I am adding a prefix to the tokens to distinguish between tags and genres.
    prefixed = [f"t:{t}" for t in tt] + [f"g:{g}" for g in gg]
    # Here, I am joining the prefixed tokens into a single string document.
    content_docs.append(" ".join(prefixed))
# Here, I am printing the number of items missing tags or genres.
print(f"Items missing tags: {miss_tags:,}/{len(all_items_universe):,}; missing genres: {miss_gen:,}/{len(all_items_universe):,}")

# Here, I am creating a TfidfVectorizer instance.
vec = TfidfVectorizer(token_pattern=r"(?u)\b[\w:]+\b", lowercase=False, sublinear_tf=True, use_idf=True, norm='l2', min_df=2)
# Here, I am fitting the vectorizer to the content documents and transforming them into a sparse TF-IDF matrix.
item_content_tfidf = vec.fit_transform(content_docs).tocsr()

# Popularity prior for bias metric
# Here, I am calculating the popularity of each item based on its frequency in the training and evaluation sets.
pop_counts = train_plus_eval_df['item'].value_counts().to_dict()
# Here, I am creating a popularity score array, where the score is the count of interactions.
pop_score = np.array([pop_counts.get(int(it), 0) for it in all_items_universe], dtype=float)
# Here, I am normalizing the popularity scores to be between 0 and 1.
pop_score = pop_score / (pop_score.max() or 1.0) 


=== RQ2 / STEP 4: TF-IDF over tags/genres (for hybrid & diversity) ===
Items missing tags: 4/8,184; missing genres: 385/8,184


In [8]:
# -------------------- STEP 5: Train ItemCF baseline --------------------
# Here, I am starting the fifth step and printing a header.
print("\n=== RQ2 / STEP 5: Train ItemCF (ItemKNN) baseline ===")
# Here, I am initializing the ItemCF model with specific parameters.
itemcf = ItemCF(
    # Here, I am setting the task to 'ranking'.
    task="ranking",
    # Here, I am providing the data information for the model.
    data_info=icf_datainfo,
    # Here, I am setting the similarity metric to cosine similarity.
    sim_type="cosine",
    # Here, I am setting the number of similar items to consider.
    k_sim=50,
    # Here, I am telling the model to store the top-k similar items.
    store_top_k=True,
    # Here, I am not using block-based training for simplicity.
    block_size=None,
    # Here, I am using all available CPU cores.
    num_threads=CPU_CORES,
    # Here, I am setting the minimum number of common users between two items for similarity calculation.
    min_common=1,
    # Here, I am using the 'invert' mode for the similarity matrix.
    mode="invert",
    # Here, I am setting the random seed for reproducibility.
    seed=SEED,
)
# Here, I am training the ItemCF model using the training data with negative sampling.
itemcf.fit(icf_train_data, neg_sampling=True, verbose=0)      



=== RQ2 / STEP 5: Train ItemCF (ItemKNN) baseline ===
Training start time: 2025-08-23 00:08:07
Final block size and num: (8184, 1)
sim_matrix elapsed: 2.630s
sim_matrix, shape: (8184, 8184), num_elements: 23022522, density: 34.3733 %


top_k: 100%|██████████| 8184/8184 [00:08<00:00, 953.19it/s] 


In [9]:
# -------------------- STEP 6: DeepFM hyperparameter search (small, safe) ----
# Here, I am starting the sixth step and printing a header.
print("\n=== RQ2 / STEP 6: Hyperparameter Tuning for DeepFM (ranking) ===")
# Here, I am creating a dictionary that maps each user to a list of items they have interacted with in the evaluation set.
eval_truth = eval_df.groupby('user')['item'].apply(list).to_dict()
# Here, I am defining a grid of hyperparameters to search over for DeepFM.
DFM_PARAM_GRID = {
    "embed_size": [16, 32],
    "hidden_units": [(128, 64), (64, 32)],
    "lr": [1e-3, 5e-4],
    "num_neg": [3],
    "n_epochs": [10, 15],
    "dropout_rate": [None, 0.2],
}
# Here, I am initializing variables to keep track of the best model and its score.
best_dfm = None
best_score = -np.inf
# Here, I am setting a limit on the number of users to sample for evaluation to keep the process fast.
EVAL_SAMPLE_USERS = min(2000, len(eval_truth))  # keep memory/CPU in check

def recommend_eval_by_model_recommend_user(model, test_truth_map, k=K, users_sample=None):
    # Here, I am evaluating a model's performance on a set of users using its `recommend_user` method.
    # Here, I am filtering for users who are in both the test truth set and the model's user ID space.
    users = [u for u in test_truth_map.keys() if int(u) in model.data_info.user2id]
    # Here, I am sampling a subset of users if a sample size is specified.
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED)
        users = random.sample(users, users_sample)
    # Here, I am initializing lists to store the evaluation metrics and recommended items.
    recalls, ndcgs, hrs, rec_items_all = [], [], [], []
    # Here, I am iterating through each user to get recommendations and calculate metrics.
    for u in tqdm(users, desc=f"Eval {model.__class__.__name__} (top-{k})"):
        # Here, I am getting the ground truth set of items for the current user.
        gt = set(test_truth_map.get(u, []))
        # Here, I am skipping users with no ground truth items.
        if not gt: continue
        # Here, I am getting the top-K recommendations from the model for the user.
        recs = model.recommend_user(int(u), n_rec=k, filter_consumed=True, inner_id=False)
        # Here, I am converting the recommendations to a NumPy array.
        topk = recs.get(int(u), np.array([], dtype=int)) if isinstance(recs, dict) else np.asarray(recs, dtype=int)
        # Here, I am skipping if no recommendations are returned.
        if topk.size == 0: continue
        # Here, I am calculating the recall, nDCG, and hit rate.
        r, n, hr = recall_ndcg_hr_at_k(topk, gt, k)
        # Here, I am appending the calculated metrics to the lists.
        recalls.append(r); ndcgs.append(n); hrs.append(hr)
        # Here, I am extending the list of all recommended items.
        rec_items_all.extend(topk.tolist())
    # Here, I am calculating the catalog coverage.
    cov = len(set(rec_items_all)) / float(len(all_items_universe)) if rec_items_all else 0.0
    # Here, I am returning a dictionary of the evaluation results.
    return {"model": model.__class__.__name__, "recall": np.mean(recalls) if recalls else 0.0,
            "ndcg": np.mean(ndcgs) if ndcgs else 0.0, "hitrate": np.mean(hrs) if hrs else 0.0,
            "coverage": cov}

for embed_size, hidden_units, lr, num_neg, n_epochs, dropout_rate in product(
    DFM_PARAM_GRID["embed_size"], DFM_PARAM_GRID["hidden_units"],
    DFM_PARAM_GRID["lr"], DFM_PARAM_GRID["num_neg"],
    DFM_PARAM_GRID["n_epochs"], DFM_PARAM_GRID["dropout_rate"]
):
    # Here, I am iterating through all possible combinations of hyperparameters.
    print(f"\n>>> DeepFM trial: emb={embed_size}, hidden={hidden_units}, lr={lr}, neg={num_neg}, "
          f"epochs={n_epochs}, dropout={dropout_rate}")
    try:
        # Here, I am resetting the TensorFlow graph to ensure a fresh start for each trial.
        import tensorflow as tf
        tf.compat.v1.reset_default_graph()
    except Exception:
        pass
    # Here, I am initializing a new DeepFM model with the current set of hyperparameters.
    dfm = DeepFM(
        # Here, I am setting the task to 'ranking'.
        task="ranking",
        # Here, I am providing the data info object.
        data_info=dfm_datainfo,
        # Here, I am setting the loss function to cross-entropy.
        loss_type="cross_entropy",
        # Here, I am setting the embedding size.
        embed_size=embed_size,
        # Here, I am setting the architecture of the hidden layers.
        hidden_units=hidden_units,
        # Here, I am setting the number of training epochs.
        n_epochs=n_epochs,
        # Here, I am setting the learning rate.
        lr=lr,
        # Here, I am setting the batch size.
        batch_size=2048,
        # Here, I am setting the sampler type for negative sampling.
        sampler="random",
        # Here, I am setting the number of negative samples per positive interaction.
        num_neg=num_neg,
        # Here, I am enabling batch normalization.
        use_bn=True,
        # Here, I am setting the dropout rate.
        dropout_rate=dropout_rate,
        # Here, I am passing the TensorFlow session configuration.
        tf_sess_config=TF_SESS_CONFIG,
        # Here, I am setting the random seed.
        seed=SEED,
    )
    # Here, I am fitting the model on the training data and evaluating it on the evaluation data.
    dfm.fit(dfm_train_data, neg_sampling=True, verbose=0, shuffle=True,
            eval_data=dfm_eval_data, metrics=["recall","ndcg"], k=K, eval_user_num=EVAL_SAMPLE_USERS)
    # Here, I am evaluating the model's performance on the evaluation set.
    r = recommend_eval_by_model_recommend_user(dfm, eval_truth, k=K, users_sample=EVAL_SAMPLE_USERS)
    # Here, I am printing the results of the current trial.
    print(f"Trial -> recall@{K}: {r['recall']:.4f}, ndcg@{K}: {r['ndcg']:.4f}")
    # Here, I am checking if the current model's NDCG score is better than the best score found so far.
    if r["ndcg"] > best_score:
        # Here, I am updating the best score and the best hyperparameters.
        best_score = r["ndcg"]
        best_dfm = (embed_size, hidden_units, lr, num_neg, n_epochs, dropout_rate)

# Here, I am printing the best hyperparameters found from the search.
print(f"\nBest DeepFM params by ndcg@{K}: {best_dfm}, ndcg={best_score:.4f}")  


=== RQ2 / STEP 6: Hyperparameter Tuning for DeepFM (ranking) ===

>>> DeepFM trial: emb=16, hidden=(128, 64), lr=0.001, neg=3, epochs=15, dropout=None
Training start time: 2025-08-23 00:12:05
Instructions for updating:
Colocations handled automatically by placer.


C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\libreco\layers\dense.py:31: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  net = tf.layers.batch_normalization(net, training=is_training)
C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\keras\legacy_tf_layers\normalization.py:455: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  return layer.apply(inputs, training=training)
2025-08-23 00:12:05,476 - WARNING - From C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\keras\layers\normalization\batch_normalization.py:532: _colocate_with (from tensorflow.python.framework.ops) is deprecated and will be removed in a future version.
Instructions 

total params: 1,064,845 | embedding params: 1,052,154 | network params: 12,691


Eval DeepFM (top-20): 100%|██████████| 2000/2000 [00:18<00:00, 107.70it/s]


Trial -> recall@20: 0.3104, ndcg@20: 0.2037

>>> DeepFM trial: emb=16, hidden=(128, 64), lr=0.001, neg=3, epochs=15, dropout=0.2
Training start time: 2025-08-23 00:24:51


C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\libreco\layers\dense.py:47: UserWarning: `tf.layers.dropout` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dropout` instead.
  net = tf.layers.dropout(net, dropout_rate, training=is_training)
C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\keras\legacy_tf_layers\core.py:401: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  return layer.apply(inputs, training=training)


total params: 1,064,845 | embedding params: 1,052,154 | network params: 12,691


Eval DeepFM (top-20): 100%|██████████| 2000/2000 [00:17<00:00, 111.78it/s]

Trial -> recall@20: 0.3220, ndcg@20: 0.2274

Best DeepFM params by ndcg@20: (16, (128, 64), 0.001, 3, 15, 0.2), ndcg=0.2274


In [10]:
# -------------------- STEP 7: Retrain final DeepFM on train+eval -------------
# Here, I am starting the seventh step and printing a header.
print("\n=== RQ2 / STEP 7: Retrain final DeepFM on train+eval ===")
# Here, I am building the training dataset using both the original training and evaluation data to train the final model.
dfm_train_final, dfm_datainfo_final = DatasetFeat.build_trainset(train_plus_eval_df)
try:
    # Here, I am resetting the TensorFlow graph again before training the final model.
    import tensorflow as tf
    tf.compat.v1.reset_default_graph()
except Exception:
    pass

# Here, I am initializing the final DeepFM model using the best hyperparameters found in the previous step.
deepfm = DeepFM(
    # Here, I am setting the task to 'ranking'.
    task="ranking",
    # Here, I am providing the data info for the combined training and evaluation set.
    data_info=dfm_datainfo_final,
    # Here, I am setting the loss function.
    loss_type="cross_entropy",
    # Here, I am using the best embedding size.
    embed_size=best_dfm[0],
    # Here, I am using the best hidden unit architecture.
    hidden_units=best_dfm[1],
    # Here, I am using the best number of epochs.
    n_epochs=best_dfm[4],
    # Here, I am using the best learning rate.
    lr=best_dfm[2],
    # Here, I am setting the batch size.
    batch_size=2048,
    # Here, I am setting the sampler type.
    sampler="random",
    # Here, I am using the best number of negative samples.
    num_neg=best_dfm[3],
    # Here, I am enabling batch normalization.
    use_bn=True,
    # Here, I am using the best dropout rate.
    dropout_rate=best_dfm[5],
    # Here, I am passing the TensorFlow session configuration.
    tf_sess_config=TF_SESS_CONFIG,
    # Here, I am setting the random seed.
    seed=SEED,
)
# Here, I am training the final DeepFM model on the combined training and evaluation data.
deepfm.fit(dfm_train_final, neg_sampling=True, verbose=0, shuffle=True)  


=== RQ2 / STEP 7: Retrain final DeepFM on train+eval ===
Training start time: 2025-08-23 00:44:40


C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\libreco\layers\dense.py:31: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_dependencies(tf.GraphKeys.UPDATE_OPS)` should not be used (consult the `tf.keras.layers.BatchNormalization` documentation).
  net = tf.layers.batch_normalization(net, training=is_training)
C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\keras\legacy_tf_layers\normalization.py:455: UserWarning: `layer.apply` is deprecated and will be removed in a future version. Please use `layer.__call__` method instead.
  return layer.apply(inputs, training=training)
C:\Users\GSI\Anaconda3\envs\kenneth\lib\site-packages\libreco\layers\dense.py:39: UserWarning: `tf.layers.batch_normalization` is deprecated and will be removed in a future version. Please use `tf.keras.layers.BatchNormalization` instead. In particular, `tf.control_depend

total params: 1,064,845 | embedding params: 1,052,154 | network params: 12,691


In [11]:
# -------------------- STEP 8: Build user histories & content profiles -------
# Here, I am starting the eighth step and printing a header.
print("\n=== RQ2 / STEP 8: User histories & content profiles (log-playtime weighted) ===")
# Here, I am creating a dictionary that maps each user to a set of items they have consumed (from the train+eval set).
user2hist_orig = train_plus_eval_df.groupby("user")["item"].agg(lambda x: set(map(int, x))).to_dict()

# build TF-IDF-based user content profile
from scipy import sparse
def build_profiles(interactions_df):
    # Here, I am building TF-IDF-based user content profiles, weighted by playtime.
    # Here, I am merging the interactions with the original data to get the playtime values.
    wdf = interactions_df.merge(inter_df[['user','item','playtime']], on=['user','item'], how='left')
    # Here, I am applying a log transformation to the playtime and clipping the values to a reasonable range.
    wdf['w'] = np.clip(np.log1p(wdf['playtime'].astype(float)), 0.0, 10.0)
    # Here, I am creating a dictionary to store user-item interactions with their weights.
    user_item_w = defaultdict(list)
    for _, r in wdf.iterrows():
        # Here, I am iterating through the weighted interactions.
        u, it, w = int(r['user']), int(r['item']), float(r['w'])
        # Here, I am getting the position of the item in the `all_items_universe` list.
        pos = item_universe_pos.get(it, None)
        # Here, I am adding the item's position and weight to the user's list.
        if pos is not None: user_item_w[u].append((pos, w))
    # Here, I am initializing dictionaries to store the user profiles, their norms, and the count of tagged items.
    user_prof, user_prof_norm, user_tagged_counts = {}, {}, {}
    for u, pairs in tqdm(user_item_w.items(), desc="Building user profiles"):
        # Here, I am handling users with no interactions.
        if not pairs:
            user_prof[u] = None; user_prof_norm[u] = 0.0; user_tagged_counts[u] = 0
            continue
        # Here, I am getting the indices and weights for the items the user interacted with.
        idxs = np.array([p[0] for p in pairs], dtype=int)
        ws  = np.array([p[1] for p in pairs], dtype=float)
        sw  = ws.sum()
        # Here, I am counting how many of the user's items have content tags.
        user_tagged_counts[u] = int((item_content_tfidf[idxs].getnnz(axis=1) > 0).sum())
        # Here, I am handling the case where the sum of weights is zero.
        if sw <= 0:
            user_prof[u] = None; user_prof_norm[u] = 0.0; continue
        # Here, I am calculating the user profile vector by taking a weighted sum of the item TF-IDF vectors.
        v = (item_content_tfidf[idxs].T.dot(ws) / (sw + 1e-12)).ravel()
        # Here, I am calculating the L2 norm of the user profile vector.
        n = np.linalg.norm(v)
        # Here, I am normalizing the user profile vector to have a unit length.
        if n > 0: v = v / n
        # Here, I am storing the calculated user profile and its norm.
        user_prof[u] = v; user_prof_norm[u] = float(n)
    # Here, I am returning the populated dictionaries.
    return user_prof, user_prof_norm, user_tagged_counts

# Here, I am calling the `build_profiles` function to create the user profiles.
user_prof, user_prof_norm, user_tagged_counts = build_profiles(train_plus_eval_df)

def exclude_consumed_idx(u):
    # Here, I am getting a list of indices for items that the user has NOT consumed.
    # Here, I am getting the set of consumed items for the user.
    consumed = user2hist_orig.get(int(u), set())
    # Here, I am creating a list of indices for items that are not in the consumed set.
    allow = np.array([i for i, it in enumerate(all_items_universe) if int(it) not in consumed], dtype=int)
    # Here, I am returning the array of allowed item indices.
    return allow       


=== RQ2 / STEP 8: User histories & content profiles (log-playtime weighted) ===


Building user profiles: 100%|██████████| 53694/53694 [00:23<00:00, 2281.50it/s]


In [12]:
# -------------------- STEP 9: Scoring functions (DeepFM, Content, Pop) ------
def cf_scores_deepfm(u, top_n=CAND_CF):
    # Here, I am generating collaborative filtering scores for a user using DeepFM.
    # Here, I am checking if the user is known to the DeepFM model.
    if int(u) not in deepfm.data_info.user2id:
        # Here, I am returning an array of zeros if the user is not found.
        return np.zeros(len(all_items_universe), dtype=float)
    # Here, I am getting the top-N recommendations from DeepFM.
    recs = deepfm.recommend_user(int(u), n_rec=top_n, filter_consumed=True, inner_id=False)
    # Here, I am extracting the recommended items as a NumPy array.
    arr = recs.get(int(u), np.array([], dtype=int)) if isinstance(recs, dict) else np.asarray(recs)
    # Here, I am initializing a scores array with zeros, representing all items in the universe.
    scores = np.zeros(len(all_items_universe), dtype=float)
    for r, it in enumerate(arr[:top_n]):
        # Here, I am iterating through the top-N recommendations.
        # Here, I am getting the position of the recommended item in the universe.
        pos = item_universe_pos.get(int(it), None)
        # Here, I am assigning a score based on reciprocal rank.
        if pos is not None:
            scores[pos] = 1.0 / float(r + 1)  # reciprocal-rank shaped
    # Here, I am returning the scores for all items.
    return scores

def cb_scores_user(u):
    # Here, I am calculating content-based scores for a user.
    # Here, I am getting the user's content profile vector.
    v = user_prof.get(int(u))
    # Here, I am returning zeros if the user profile is not found.
    if v is None: return np.zeros(len(all_items_universe), dtype=float)
    # Here, I am calculating the cosine similarity between the user profile and all item content vectors.
    s = item_content_tfidf.dot(v)
    # Here, I am returning the scores as a flattened NumPy array.
    return np.asarray(s).ravel()

def pop_scores_user(u):
    # Here, I am returning the pre-calculated popularity scores for all items.
    return pop_score

def conservative_alpha(u, base_alpha):
    # Here, I am calculating an adaptive alpha value for a user based on their content profile quality.
    # Here, I am getting the L2 norm of the user's content profile and the count of tagged items they interacted with.
    prof_norm = user_prof_norm.get(int(u), 0.0)
    tagged_cnt = user_tagged_counts.get(int(u), 0)
    # Here, I am checking if the user's profile is weak (low norm or few tagged items).
    if prof_norm <= 0 or tagged_cnt < MIN_TAG_ITEMS:
        # Here, if the profile is weak, I am increasing the alpha value to rely more on collaborative filtering.
        bump = 0.5 + 0.5 * (MIN_TAG_ITEMS - min(tagged_cnt, MIN_TAG_ITEMS)) / float(max(1, MIN_TAG_ITEMS))
        return float(min(1.0, base_alpha + (1.0 - base_alpha) * bump))
    # Here, I am using a sigmoid function to create a smooth factor that penalizes low-norm profiles.
    factor = sigmoid((prof_norm - MIN_PROFILE_NORM) * ADAPT_S)
    # Here, I am adjusting the base alpha by the factor, resulting in a more conservative alpha for strong profiles.
    return float(base_alpha * (0.5 + 0.5 * factor))

def rrf_blend_from_scores(cf_scores, cb_scores, alpha):
    # Here, I am blending collaborative filtering and content-based scores using Reciprocal Rank Fusion.
    # Here, I am converting the collaborative filtering scores to RRF scores.
    cf_rrf = score_to_rrf(cf_scores)
    # Here, I am converting the content-based scores to RRF scores.
    cb_rrf = score_to_rrf(cb_scores)
    # Here, I am returning the blended scores using the given alpha.
    return alpha * cf_rrf + (1.0 - alpha) * cb_rrf

In [13]:
# -------------------- STEP 10: Evaluation wrappers (consistent) --------------
# Here, I am creating a dictionary of the ground truth items for the test set.
test_truth = test_df.groupby('user')['item'].apply(list).to_dict()
# Here, I am calculating the popularity (frequency) of each item in the train+eval set.
train_item_freq = train_plus_eval_df.groupby("item")["user"].count()
# Here, I am creating an array of item frequencies.
all_freqs = np.array([train_item_freq.get(int(it), 0) for it in all_items_universe], dtype=float)
if all_freqs.max() > 0:
    # Here, I am calculating the rank of each item based on its popularity.
    pop_rank = np.argsort(np.argsort(-all_freqs))
    # Here, I am converting the popularity rank to a percentile (higher value means more popular).
    pop_percentile = 1.0 - (pop_rank / float(len(all_freqs)-1 + 1e-12))
else:
    # Here, I am handling the case where there are no frequencies.
    pop_percentile = np.zeros_like(all_freqs)

def eval_model_scores(get_scores_fn, label, k=K, users_sample=None):
    # Here, I am evaluating a model by using a scoring function to get scores for all items.
    # Here, I am getting a list of users to evaluate.
    users = list(test_truth.keys())
    # Here, I am sampling a subset of users if a sample size is specified.
    if users_sample is not None and len(users) > users_sample:
        random.seed(SEED); users = random.sample(users, users_sample)
    # Here, I am initializing lists to store evaluation metrics.
    recalls, ndcgs, hrs, rec_items_all = [], [], [], []
    diversity_list, pop_percentiles = [], []
    for u in tqdm(users, desc=f"Eval scores {label} (top-{k})"):
        # Here, I am getting the ground truth set of items for the user.
        gt = set(test_truth.get(u, []))
        # Here, I am skipping users with no ground truth items.
        if not gt: continue
        # Here, I am getting the indices of items the user has not consumed.
        allow = exclude_consumed_idx(u)
        # Here, I am skipping if there are no items to recommend.
        if allow.size == 0: continue
        # Here, I am getting the scores for all items from the scoring function.
        scores = get_scores_fn(u)
        # Here, I am getting the top-K recommended item indices based on the scores.
        idxs = topk_from_scores(scores, allow, k)
        # Here, I am skipping if no recommendations are returned.
        if idxs.size == 0: continue
        # Here, I am getting the original item IDs from the indices.
        topk_items = [all_items_universe[i] for i in idxs]
        # Here, I am calculating the ranking metrics (recall, nDCG, hit rate).
        r, n, hr = recall_ndcg_hr_at_k(topk_items, gt, k)
        # Here, I am appending the calculated metrics.
        recalls.append(r); ndcgs.append(n); hrs.append(hr)
        # Here, I am extending the list of all recommended items.
        rec_items_all.extend(topk_items)
        try:
            # Here, I am calculating the intra-list diversity.
            rows = item_content_tfidf[idxs]
            diversity_list.append(pairwise_cosine_from_tfidf(rows))
        except Exception:
            pass
        # Here, I am calculating the average popularity percentile of the recommended items.
        pop_percentiles.append(pop_percentile[idxs].mean() if idxs.size > 0 else 0.0)
    # Here, I am returning a dictionary of the final evaluation results.
    return {
        "model": label,
        "recall": float(np.mean(recalls) if recalls else 0.0),
        "ndcg": float(np.mean(ndcgs) if ndcgs else 0.0),
        "hitrate": float(np.mean(hrs) if hrs else 0.0),
        "coverage": len(set(rec_items_all)) / float(len(all_items_universe)) if rec_items_all else 0.0,
        "diversity": float(np.mean(diversity_list) if diversity_list else 0.0),
        "pop_bias": float(np.mean(pop_percentiles) if pop_percentiles else 0.0),
    }            

In [14]:
# -------------------- STEP 11: Evaluate baselines & DeepFM & Hybrid ----------
# Here, I am starting the eleventh step and printing a header.
print("\n=== RQ2 / STEP 11: Evaluate ItemCF & Popularity (score-based) ===")
# Here, I am evaluating the Popularity baseline model.
pop_res = eval_model_scores(pop_scores_user, "Popularity", k=K)
# Here, I am evaluating the ItemCF model using its `recommend_user` method.
icf_res = recommend_eval_by_model_recommend_user(itemcf, test_truth, k=K, users_sample=None)

print("\n=== RQ2 / STEP 11b: Evaluate DeepFM (collaborative) ===")
# Evaluate DeepFM by using its recommend_user directly (canonical)
# Here, I am evaluating the DeepFM model using its `recommend_user` method.
dfm_res = recommend_eval_by_model_recommend_user(deepfm, test_truth, k=K, users_sample=None)

print("\n=== RQ2 / STEP 11c: DeepFM+Tags Hybrid ===")
# α-sweep (global alpha), adaptive per-user inside (conservative_alpha)
# Here, I am starting the alpha sweep to find the best blending weight for the hybrid model.
alpha_grid_rows = []
for a in ALPHAS_GRID:
    # Here, I am iterating through each alpha value.
    # Here, I am evaluating the hybrid model with the current alpha, using the `conservative_alpha` function.
    res = eval_model_scores(
        lambda u: rrf_blend_from_scores(cf_scores_deepfm(u), cb_scores_user(u), conservative_alpha(u, a)),
        f"DeepFM+Tags (a={a:.2f})", k=K
    )
    # Here, I am adding the current alpha value to the results.
    res["alpha"] = float(a)
    # Here, I am appending the results to the list.
    alpha_grid_rows.append(res)
# Here, I am converting the list of results into a pandas DataFrame.
alpha_df = pd.DataFrame(alpha_grid_rows)
# Here, I am saving the alpha sweep results to a CSV file.
alpha_df.to_csv(EVAL_ALPHA_GRID_OUT, index=False)
# Here, I am finding the alpha value that gives the best recall and nDCG.
best_alpha = float(alpha_df.sort_values(["recall","ndcg"], ascending=False).iloc[0]["alpha"])
# Here, I am printing the best alpha value found.
print(f"Best α from sweep: {best_alpha:.2f}")

# Here, I am evaluating the final hybrid model using the best alpha found.
hyb_res = eval_model_scores(
    lambda u: rrf_blend_from_scores(cf_scores_deepfm(u), cb_scores_user(u), conservative_alpha(u, best_alpha)),
    "DeepFM+Tags (best-α)", k=K
)          


=== RQ2 / STEP 11: Evaluate ItemCF & Popularity (score-based) ===


Eval ItemCF (top-20): 100%|██████████| 42071/42071 [04:16<00:00, 163.93it/s]



=== RQ2 / STEP 11b: Evaluate DeepFM (collaborative) ===


Eval DeepFM (top-20): 100%|██████████| 42071/42071 [07:12<00:00, 97.16it/s] 



=== RQ2 / STEP 11c: DeepFM+Tags Hybrid ===


Eval scores DeepFM+Tags (a=0.95) (top-20): 100%|██████████| 42071/42071 [16:54<00:00, 41.46it/s]


Best α from sweep: 0.90


Eval scores DeepFM+Tags (best-α) (top-20): 100%|██████████| 42071/42071 [16:57<00:00, 41.36it/s]


In [15]:
# -------------------- STEP 12: Summary table & save --------------------------
# Here, I am starting the twelfth step and printing a header.
# Here, I am preparing a list of dictionaries, one for each model, to create the summary table.
rows = [
    {"model":"DeepFM", **{k:v for k,v in dfm_res.items() if k!='model'}},
    {"model":"DeepFM+Tags (best-α)", **{k:v for k,v in hyb_res.items() if k!='model'}},
    {"model":"ItemCF", **{k:v for k,v in icf_res.items() if k!='model'}},
    {"model":"Popularity", **{k:v for k,v in pop_res.items() if k!='model'}},
]
# Here, I am creating a DataFrame from the list of results.
summary_df = pd.DataFrame(rows)[["model","recall","ndcg","hitrate","coverage","diversity","pop_bias"]]
# Here, I am sorting the summary table by recall and nDCG in descending order.
summary_df = summary_df.sort_values(["recall","ndcg"], ascending=False).reset_index(drop=True)
# Here, I am printing the final summary table.
print("\nRQ2 Evaluation summary:\n", summary_df)
# Here, I am saving the summary table to a CSV file.
summary_df.to_csv(EVAL_SUMMARY_OUT, index=False)
# Here, I am confirming that the summary and alpha sweep files have been saved.
print(f"Saved: {EVAL_SUMMARY_OUT}\nSaved: {EVAL_ALPHA_GRID_OUT}")   


RQ2 Evaluation summary:
                   model    recall      ndcg   hitrate  coverage  diversity  \
0  DeepFM+Tags (best-α)  0.339809  0.233294  0.728221  0.262097   0.711387   
1                DeepFM  0.335763  0.245972  0.723729  0.264174        NaN   
2                ItemCF  0.301522  0.218563  0.687100  0.175342        NaN   
3            Popularity  0.272171  0.202940  0.643317  0.010386   0.735618   

   pop_bias  
0  0.988204  
1       NaN  
2       NaN  
3  0.998047  
Saved: C:\Users\GSI\Pictures\DATASETS\rq2_eval_summary.csv
Saved: C:\Users\GSI\Pictures\DATASETS\rq2_alpha_sweep.csv


In [16]:
# -------------------- STEP 13: Plots (accuracy, coverage/diversity, pop bias, alpha sweep) ----
# Here, I am starting the thirteenth step and printing a header.
print("\n=== RQ2 / STEP 13: Plots ===")
# A) Accuracy bars
# Here, I am creating a new figure for the accuracy bar plot.
plt.figure(figsize=(9,5))
# Here, I am creating an array of x-coordinates for the bars.
x = np.arange(len(summary_df))
# Here, I am plotting the recall and nDCG bars.
plt.bar(x-0.2, summary_df["recall"].values, width=0.4, label=f"Recall@{K}")
plt.bar(x+0.2, summary_df["ndcg"].values,   width=0.4, label=f"NDCG@{K}")
# Here, I am setting the x-axis tick labels to the model names.
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
# Here, I am setting the y-axis label, title, and legend.
plt.ylabel("Score"); plt.title("RQ2: Top-K Ranking Quality"); plt.legend()
# Here, I am adjusting the plot layout and saving the figure.
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rq2_accuracy_bars.png")); plt.close()

# B) Coverage & Diversity
# Here, I am creating a new figure for the coverage and diversity bar plot.
plt.figure(figsize=(9,5))
# Here, I am plotting the coverage and diversity bars.
plt.bar(x-0.2, summary_df["coverage"].values, width=0.4, label=f"Catalog Coverage@{K}")
plt.bar(x+0.2, summary_df["diversity"].fillna(0).values, width=0.4, label=f"Intra-List Diversity@{K}")
# Here, I am setting the x-axis tick labels.
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
# Here, I am setting the y-axis label, title, and legend.
plt.ylabel("Score"); plt.title("RQ2: Coverage & Diversity Trade-offs"); plt.legend()
# Here, I am adjusting the plot layout and saving the figure.
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rq2_coverage_diversity_bars.png")); plt.close()

# C) Popularity bias
# Here, I am creating a new figure for the popularity bias bar plot.
plt.figure(figsize=(9,5))
# Here, I am plotting the popularity bias bars.
plt.bar(x, summary_df["pop_bias"].fillna(0).values)
# Here, I am setting the x-axis tick labels.
plt.xticks(x, summary_df["model"].values, rotation=15, ha="right")
# Here, I am setting the y-axis label and title.
plt.ylabel("Avg popularity percentile (higher ⇒ more head)"); plt.title("RQ2: Popularity Bias")
# Here, I am adjusting the plot layout and saving the figure.
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rq2_pop_bias.png")); plt.close()

# D) α-sweep curves
# Here, I am checking if the alpha sweep data exists.
if not alpha_df.empty:
    # Here, I am creating a new figure for the alpha sweep plot.
    plt.figure(figsize=(8,5))
    # Here, I am plotting the recall and nDCG curves as a function of alpha.
    plt.plot(alpha_df["alpha"].values, alpha_df["recall"].values, marker='o', label=f"Recall@{K}")
    plt.plot(alpha_df["alpha"].values, alpha_df["ndcg"].values,   marker='s', label=f"NDCG@{K}")
    # Here, I am setting the x and y axis labels, title, grid, and legend.
    plt.xlabel("Alpha (CF weight)"); plt.ylabel("Score")
    plt.title("RQ2: DeepFM+Tags α-sweep"); plt.grid(True); plt.legend()
    # Here, I am adjusting the plot layout and saving the figure.
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "rq2_hybrid_alpha_sweep.png")); plt.close()

# Here, I am confirming that the plots have been saved.
print("Saved RQ2 plots to:", PLOT_DIR)   


=== RQ2 / STEP 13: Plots ===
Saved RQ2 plots to: C:\Users\GSI\Pictures\DATASETS


In [17]:
# -------------------- STEP 14: Export Top-20 for test_interactions_in.csv ----
# Here, I am starting the fourteenth step and printing a header.
print("\n=== RQ2 / STEP 14: Export top-20 for test_interactions_in.csv (DeepFM+Tags best-α) ===")
# Here, I am checking if the test interactions file exists.
if test_in_df is None:
    # Here, if the file doesn't exist, I am skipping this step.
    print("No test_interactions_in.csv found; skipping final recommendations.")
else:
    # Here, if the file exists, I am proceeding with generating recommendations.
    # Here, I am initializing a list to store the output rows.
    out_rows = []
    # Here, I am iterating through each unique user in the test interactions file.
    for u in tqdm(test_in_df["user"].unique(), desc="Inferring RQ2 top-20"):
        # Here, I am getting the indices of items the user has not consumed.
        allow = exclude_consumed_idx(u)
        # Here, I am calculating the adaptive alpha for the current user using the best global alpha.
        a_u = conservative_alpha(u, best_alpha)
        # Here, I am getting the scores from the collaborative filtering model.
        cf_all = cf_scores_deepfm(u)
        # Here, I am getting the scores from the content-based model.
        cb_all = cb_scores_user(u)
        # Here, I am blending the CF and CB scores using Reciprocal Rank Fusion.
        blended = rrf_blend_from_scores(cf_all, cb_all, a_u)
        # Here, I am getting the top-K recommended item indices.
        idxs = topk_from_scores(blended, allow, K)
        # Here, I am iterating through the top recommended item indices.
        for pos in idxs:
            # Here, I am appending the user ID and the recommended item ID to the output list.
            out_rows.append((int(u), int(all_items_universe[pos])))
    # Here, I am creating a DataFrame from the output rows.
    out_df = pd.DataFrame(out_rows, columns=["user_id","item_id"])
    # Here, I am saving the final recommendations to a CSV file.
    out_df.to_csv(OUTPUT_TOP20, index=False)
    # Here, I am confirming that the file has been saved and showing the number of recommendations.
    print(f"Saved top-{K} recommendations to {OUTPUT_TOP20} ({len(out_df):,} rows)")


=== RQ2 / STEP 14: Export top-20 for test_interactions_in.csv (DeepFM+Tags best-α) ===


Inferring RQ2 top-20: 100%|██████████| 13579/13579 [00:40<00:00, 336.68it/s]


Saved top-20 recommendations to C:\Users\GSI\Pictures\DATASETS\rq2_top20_deepfm_hybrid.csv (271,580 rows)
